# Tính Recall@5 và NDCG@5 từ 10 clickout có điểm do model xuất
Notebook nạp JSON, tự dựng dataset và kiểm tra thứ hạng mục tiêu trước khi tính metric.

In [1]:
import numpy as np

# Adapter NumPy nhỏ để chạy nguyên hàm train trên máy chưa cài PyTorch.
class DemoTensor:
    def __init__(self, data):
        self.data = np.asarray(data)
        self.device = 'cpu'
    def __getitem__(self, index):
        return DemoTensor(self.data[index])
    def item(self):
        return self.data.item()
    def index_select(self, dim, indices):
        return DemoTensor(np.take(self.data, indices.data, axis=dim))
    def __gt__(self, other):
        return DemoTensor(self.data > getattr(other, 'data', other))
    def eq(self, other):
        return DemoTensor(self.data == getattr(other, 'data', other))
    def lt(self, other):
        return DemoTensor(self.data < getattr(other, 'data', other))
    def __and__(self, other):
        return DemoTensor(self.data & other.data)
    def sum(self):
        return DemoTensor(self.data.sum())

class DemoTorch:
    long = np.int64
    float32 = np.float32
    @staticmethod
    def tensor(data, dtype=None):
        return DemoTensor(np.asarray(data, dtype=dtype))
    @staticmethod
    def as_tensor(data, device=None, dtype=None):
        return DemoTensor(np.asarray(data, dtype=dtype))
    @staticmethod
    def arange(n, device=None):
        return DemoTensor(np.arange(n))

torch = DemoTorch()

# Alias phục vụ type annotations giống file train.
MLEUP = object
Data = list

class DemoModel:
    def eval(self):
        return self

def batch_at(dataset, indices):
    return [dataset[i] for i in indices]

def forward(model, batch):
    targets = torch.tensor([row['target_position'] for row in batch], dtype=torch.long)
    # Số ứng viên mỗi clickout có thể khác nhau; pad thành ma trận chữ nhật.
    # Padding là -inf và evaluate chỉ xếp hạng các ứng viên thật.
    max_candidates = max(len(row['scores']) for row in batch)
    padded_scores = np.full((len(batch), max_candidates), -np.inf, dtype=np.float32)
    for index, row in enumerate(batch):
        row_scores = np.asarray(row['scores'], dtype=np.float32)
        padded_scores[index, :len(row_scores)] = row_scores
    scores = torch.tensor(padded_scores, dtype=torch.float32)
    return targets, scores, None

def check_time(started, max_hours):
    return None  # Không giới hạn thời gian trong bộ dữ liệu demo.


In [2]:
# Nạp 10 clickout và điểm ứng viên đã xuất từ model.
from pathlib import Path
import json
import numpy as np

JSON_NAME = 'metric_sample_epoch1_10.json'
search_roots = [Path.cwd(), Path(r'C:\Users\Admin\Desktop\Test')]
if Path('/kaggle/input').is_dir():
    search_roots.append(Path('/kaggle/input'))

json_matches = []
for root in search_roots:
    if root.is_dir():
        try:
            json_matches.extend(p for p in root.rglob(JSON_NAME) if p.is_file())
        except (PermissionError, OSError):
            continue
json_matches = list(dict.fromkeys(p.resolve() for p in json_matches))
if len(json_matches) != 1:
    raise FileNotFoundError(
        f'Cần tìm đúng một {JSON_NAME}; tìm thấy {len(json_matches)} file: '
        f'{json_matches}. Đặt JSON cạnh notebook hoặc thêm vào Kaggle Input.'
    )
JSON_PATH = json_matches[0]
with JSON_PATH.open('r', encoding='utf-8') as source:
    raw_records = json.load(source)

required_keys = {
    'click_id', 'target_id', 'target_position',
    'candidate_ids', 'candidate_scores', 'target_rank',
}
if not isinstance(raw_records, list) or not raw_records:
    raise ValueError('JSON phải là danh sách clickout không rỗng.')

DATASET = []
CANDIDATES = []
for index, record in enumerate(raw_records):
    missing = required_keys - record.keys()
    if missing:
        raise ValueError(f'Record {index} thiếu trường: {sorted(missing)}')
    ids = record['candidate_ids']
    scores = record['candidate_scores']
    position = int(record['target_position'])
    if len(ids) != len(scores) or not scores:
        raise ValueError(f'Record {index}: candidate_ids/scores không khớp.')
    if not 0 <= position < len(ids) or ids[position] != record['target_id']:
        raise ValueError(f'Record {index}: target_position không trỏ tới target_id.')

    target_score = float(scores[position])
    rank = 1 + sum(float(score) > target_score for score in scores)
    rank += sum(float(score) == target_score for score in scores[:position])
    if rank != int(record['target_rank']):
        raise ValueError(
            f"Record {index}: rank tính từ điểm ({rank}) khác target_rank "
            f"trong JSON ({record['target_rank']})."
        )

    # Điểm trong mỗi hàng được xếp theo thứ tự candidate_ids của chính clickout đó.
    DATASET.append({
        'session_id': record['click_id'],
        'target_position': position,
        'scores': [float(value) for value in scores],
        'target_score': target_score,
        'target_rank': rank,
        'target_id': record['target_id'],
        'candidate_ids': ids,
    })
    CANDIDATES.append(list(range(len(ids))))

# Các chỉ số trong candidate_scores là vị trí cục bộ của từng clickout.
print('JSON:', JSON_PATH)

print('\nClickout | target_id | rank | target_score | NDCG@5 | Recall@5')
for row in DATASET:
    rank = row['target_rank']
    ndcg = 1 / np.log2(rank + 1) if rank <= 5 else 0.0
    recall = int(rank <= 5)
    print(
        f"{row['session_id']:>8} | {row['target_id']:>9} | {rank:>4} | "
        f"{row['target_score']:>12.2f} | {ndcg:>6.2f} | {recall:>8}"
    )

JSON: C:\Users\Admin\Desktop\Test\metric_sample_epoch1_10.json

Clickout | target_id | rank | target_score | NDCG@5 | Recall@5
  121513 |     19356 |    6 |         8.25 |   0.00 |        0
    1315 |    926852 |   10 |         1.28 |   0.00 |        0
   46043 |    424151 |    1 |         5.68 |   1.00 |        1
  106673 |    645894 |    3 |         7.80 |   0.50 |        1
  104292 |    534362 |    1 |         6.90 |   1.00 |        1
  104599 |     76212 |    2 |         3.28 |   0.63 |        1
  118556 |    340321 |    1 |         6.64 |   1.00 |        1
  106135 |    228798 |   10 |         4.00 |   0.00 |        0
   96289 |     62670 |   11 |         2.64 |   0.00 |        0
     216 |    487170 |    3 |         7.22 |   0.50 |        1


Cell 2 nạp điểm ứng viên do model xuất, dựng DATASET và kiểm tra target_rank có khớp với candidate_scores. Các kiểm tra được giữ nhưng không in thống kê tổng hợp; bảng metric từng clickout được hiển thị bên dưới.

In [3]:
def evaluate(model: MLEUP, dataset: Data, candidates: list, batch_size: int,
             started: float = 0, max_hours: float = 0) -> dict:
    if len(candidates) != len(dataset):
        raise ValueError("Candidate lists must align with the evaluation examples")
    model.eval()
    count = 0
    recall5 = 0.0
    ndcg5 = 0.0
    for offset in range(0, len(dataset), batch_size):
        indices = list(range(offset, min(offset + batch_size, len(dataset))))
        targets, scores, _ = forward(model, batch_at(dataset, indices))
        for row, source_index in enumerate(indices):
            item_ids = candidates[source_index]
            target_id = int(targets[row].item())
            target_position = item_ids.index(target_id)
            positions = torch.as_tensor(item_ids, device=scores.device, dtype=torch.long)
            values = scores[row].index_select(0, positions)
            target_score = values[target_position]
            before = torch.arange(len(item_ids), device=scores.device).lt(target_position)
            rank = 1 + (values > target_score).sum().item() + (
                values.eq(target_score) & before
            ).sum().item()
            count += 1
            # One relevant target per clickout: Recall@5 equals Hit@5.
            recall5 += int(rank <= 5)
            ndcg5 += 1.0 / np.log2(rank + 1) if rank <= 5 else 0.0
        if offset % (batch_size * 500) == 0:
            print(f"  eval: {offset:,}/{len(dataset):,}", flush=True)
        if started:
            check_time(started, max_hours)
    if not count:
        raise ValueError("Cannot evaluate an empty split")
    return {"ndcg_at_5": ndcg5 / count,
            "recall_at_5": recall5 / count,
            "examples": count}

In [4]:
# Tính metric bằng đúng hàm evaluate dùng trong code train.
metrics = evaluate(DemoModel(), DATASET, CANDIDATES, batch_size=4)
print('\nKết quả trên 10 clickout:')
print(f"NDCG@5: {metrics['ndcg_at_5']:.2f}")
print(f"Recall@5: {metrics['recall_at_5']:.2f}")
print(f"Số clickout: {metrics['examples']}")

  eval: 0/10

Kết quả trên 10 clickout:
NDCG@5: 0.46
Recall@5: 0.60
Số clickout: 10
